In [5]:
# ============================================================
# CELL 1: KAGGLE SETUP
# ============================================================
!pip -q install timm opencv-python scikit-learn

from __future__ import annotations

import csv
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from tqdm.auto import tqdm

# Kaggle-friendly directories
KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
MODELS_DIR = WORK_ROOT / 'models'
OUTPUTS_DIR = WORK_ROOT / 'outputs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = torch.cuda.is_available()
print(f'Using device: {device}')

# Training configuration
CFG = {
    'seed': 42,
    'seq_len': 12,
    'img_size': 299,
    'batch_size': 2 if torch.cuda.is_available() else 1,
    'num_workers': 0,  # 0 is most stable on Kaggle notebooks
    'epochs': 7,
    'freeze_backbone_epochs': 1,
    'val_size': 0.15,
    'test_size': 0.15,
    'max_per_class': 0,
    'optimizer_name': 'adamw',  # options: adamw, adam, adagrad
    'lr': 5e-5,
    'weight_decay': 1e-4,
    'pos_weight': 2.0,
    'threshold': 0.5,
    'model_name': 'video_xception_bilstm_kaggle.pth',
}

print(json.dumps(CFG, indent=2))

Using device: cuda
{
  "seed": 42,
  "seq_len": 12,
  "img_size": 299,
  "batch_size": 2,
  "num_workers": 0,
  "epochs": 7,
  "freeze_backbone_epochs": 1,
  "val_size": 0.15,
  "test_size": 0.15,
  "max_per_class": 0,
  "optimizer_name": "adamw",
  "lr": 5e-05,
  "weight_decay": 0.0001,
  "pos_weight": 2.0,
  "threshold": 0.5,
  "model_name": "video_xception_bilstm_kaggle.pth"
}


In [6]:
# ============================================================
# CELL 1B: GPU SANITY CHECK
# ============================================================
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU detected: {gpu_name}')
    print(f'GPU VRAM: {total_mem_gb:.2f} GB')
else:
    print('CUDA not available. Training will run on CPU.')
print(f'Model/training device target: {device}')

GPU detected: Tesla T4
GPU VRAM: 14.56 GB
Model/training device target: cuda


In [ ]:
# ============================================================
# CELL 2: DATASET DISCOVERY AND LOADING
# ============================================================
@dataclass
class VideoSample:
    path: Path
    label: int

def _discover_candidate_roots() -> list[Path]:
    candidates: list[Path] = []

    env_value = os.getenv('DEEPFAKE_VIDEO_DATASET_ROOT') or os.getenv('DEEPFAKE_DATASET_ROOT')
    if env_value:
        candidates.append(Path(env_value))

    # Kaggle uploads usually live in /kaggle/input/<dataset-name>/...
    if KAGGLE_INPUT_ROOT.exists():
        candidates.append(KAGGLE_INPUT_ROOT)
        for child in KAGGLE_INPUT_ROOT.iterdir():
            if child.is_dir():
                candidates.append(child)
                for nested in child.iterdir():
                    if nested.is_dir():
                        candidates.append(nested)
                        for nested2 in nested.iterdir():
                            if nested2.is_dir():
                                candidates.append(nested2)

    candidates.append(Path.cwd())
    candidates.append(Path('/kaggle/working'))

    # De-duplicate while preserving order
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in candidates:
        rp = p.resolve() if p.exists() else p
        if rp not in seen:
            seen.add(rp)
            unique.append(p)
    return unique

def _has_video_files(folder: Path) -> bool:
    if not folder.exists() or not folder.is_dir():
        return False
    for ext in ('*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm'):
        if next(folder.rglob(ext), None) is not None:
            return True
    return False

def find_dataset_root() -> Path:
    marker_dirs = (
        'ff_face_only_data',
        'celeb_fake_face_only',
        'celeb_real_face_only',
        'dfdc_fake_face_only_data',
        'dfdc_real_face_only_data',
    )

    for base in _discover_candidate_roots():
        if not base.exists():
            continue

        # 1) Direct known layout
        base_children = {p.name.lower() for p in base.iterdir() if p.is_dir()}
        if any(name in base_children for name in marker_dirs):
            return base

        # 2) Legacy typo layout container
        if (base / 'vedio_data').exists():
            return base / 'vedio_data'

        # 3) If this folder itself looks like a dataset root with videos and labels
        if _has_video_files(base):
            lower_parts = [part.lower() for part in base.parts]
            looks_like_df = any(k in '/'.join(lower_parts) for k in ('deepfake', 'dfdc', 'face_only', 'ff_'))
            if looks_like_df or (base / 'metadata.csv').exists():
                return base

    raise FileNotFoundError(
        'Could not locate Kaggle video dataset automatically. '\
        'Set DEEPFAKE_VIDEO_DATASET_ROOT to your dataset folder, e.g. /kaggle/input/<dataset-name>.'
    )

DATASET_ROOT = find_dataset_root()
print(f'Dataset root: {DATASET_ROOT}')

def iter_videos(folder: Path):
    for ext in ('*.mp4', '*.avi', '*.mov', '*.mkv', '*.webm'):
        yield from folder.rglob(ext)

def parse_ff_metadata(path: Path) -> dict[str, int]:
    mapping: dict[str, int] = {}
    if not path.exists():
        return mapping
    with path.open('r', encoding='utf-8') as f:
        reader = csv.reader(f)
        for row in reader:
            if len(row) < 2:
                continue
            name = row[0].strip()
            label_text = row[1].strip().upper()
            if not name or label_text == 'LABEL':
                continue
            if label_text == 'FAKE':
                mapping[name] = 1
            elif label_text == 'REAL':
                mapping[name] = 0
    return mapping

def gather_samples(dataset_root: Path) -> list[VideoSample]:
    samples: list[VideoSample] = []

    celeb_fake = dataset_root / 'Celeb_fake_face_only' / 'Celeb_fake_face_only'
    celeb_real = dataset_root / 'Celeb_real_face_only' / 'Celeb_real_face_only'
    dfdc_fake = dataset_root / 'DFDC_FAKE_Face_only_data' / 'DFDC_FAKE_Face_only_data'
    dfdc_real = dataset_root / 'DFDC_REAL_Face_only_data' / 'DFDC_REAL_Face_only_data'
    ff_dir = dataset_root / 'FF_Face_only_data' / 'FF_Face_only_data'

    for root, label in ((celeb_fake, 1), (dfdc_fake, 1), (celeb_real, 0), (dfdc_real, 0)):
        if root.exists():
            for p in iter_videos(root):
                samples.append(VideoSample(path=p, label=label))

    ff_map = parse_ff_metadata(ff_dir / 'metadata.csv')
    if ff_dir.exists() and ff_map:
        for p in iter_videos(ff_dir):
            label = ff_map.get(p.name)
            if label is not None:
                samples.append(VideoSample(path=p, label=label))

    if samples:
        return samples

    # Fallback: infer labels from folder names containing 'real' or 'fake'
    for p in iter_videos(dataset_root):
        parts = [segment.lower() for segment in p.parts]
        has_real = any('real' in segment for segment in parts)
        has_fake = any('fake' in segment for segment in parts)
        if has_real and not has_fake:
            samples.append(VideoSample(path=p, label=0))
        elif has_fake and not has_real:
            samples.append(VideoSample(path=p, label=1))

    return samples

all_samples = gather_samples(DATASET_ROOT)
if not all_samples:
    raise RuntimeError('No labeled videos found in the dataset root.')

print(f'Total labeled videos found: {len(all_samples)}')
print(f'Fake videos: {sum(s.label for s in all_samples)}')
print(f'Real videos: {len(all_samples) - sum(s.label for s in all_samples)}')

labels = np.array([s.label for s in all_samples], dtype=np.int64)
indices = np.arange(len(all_samples))

holdout_size = CFG['val_size'] + CFG['test_size']
splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=holdout_size, random_state=CFG['seed'])
train_idx, holdout_idx = next(splitter1.split(indices, labels))

holdout_labels = labels[holdout_idx]
val_fraction_of_holdout = CFG['val_size'] / holdout_size
splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=(1 - val_fraction_of_holdout), random_state=CFG['seed'])
val_rel_idx, test_rel_idx = next(splitter2.split(holdout_idx, holdout_labels))

val_idx = holdout_idx[val_rel_idx]
test_idx = holdout_idx[test_rel_idx]

train_samples = [all_samples[i] for i in train_idx]
val_samples = [all_samples[i] for i in val_idx]
test_samples = [all_samples[i] for i in test_idx]

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

train_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

Dataset root: /kaggle/input/datasets/sartham811/vedio-data/vedio_data


In [8]:
# ============================================================
# CELL 3: DATASET + DATALOADERS
# ============================================================
def extract_even_frames(video_path: Path, seq_len: int) -> list[Image.Image]:
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames: list[Image.Image] = []

    if total_frames <= 0:
        cap.release()
        return frames

    if total_frames < seq_len:
        indices = list(range(total_frames))
        while indices and len(indices) < seq_len:
            indices.extend(indices[: seq_len - len(indices)])
    else:
        indices = np.linspace(0, total_frames - 1, seq_len, dtype=int).tolist()

    for idx in indices[:seq_len]:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(Image.fromarray(frame_rgb))

    cap.release()
    return frames

class VideoDeepfakeDataset(Dataset):
    def __init__(self, samples: list[VideoSample], seq_len: int, transform):
        self.samples = samples
        self.seq_len = seq_len
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        frames = extract_even_frames(sample.path, self.seq_len)

        if not frames:
            frame_tensors = [torch.zeros(3, CFG['img_size'], CFG['img_size'])] * self.seq_len
        else:
            frame_tensors = [self.transform(frame) for frame in frames[: self.seq_len]]
            while len(frame_tensors) < self.seq_len:
                frame_tensors.append(frame_tensors[-1])

        x = torch.stack(frame_tensors)
        y = torch.tensor(sample.label, dtype=torch.long)
        return x, y

train_ds = VideoDeepfakeDataset(train_samples, CFG['seq_len'], train_transform)
val_ds = VideoDeepfakeDataset(val_samples, CFG['seq_len'], eval_transform)
test_ds = VideoDeepfakeDataset(test_samples, CFG['seq_len'], eval_transform)

train_labels = np.array([s.label for s in train_samples], dtype=np.int64)
class_counts = np.bincount(train_labels, minlength=2)
class_counts = np.maximum(class_counts, 1)
class_weights_np = (1.0 / class_counts).astype(np.float64)
sample_weights = np.array([class_weights_np[label] for label in train_labels], dtype=np.float64)
sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    sampler=sampler,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_ds,
    batch_size=CFG['batch_size'],
    shuffle=False,
    num_workers=CFG['num_workers'],
    pin_memory=torch.cuda.is_available(),
)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 2258
Val batches: 484
Test batches: 484


In [9]:
# ============================================================
# CELL 4: MODEL DEFINITION
# ============================================================
class VideoModel(nn.Module):
    def __init__(self, seq_len: int = 12, hidden_dim: int = 256, num_layers: int = 2, dropout: float = 0.5):
        super().__init__()
        self.xception = timm.create_model('xception', pretrained=True)
        if hasattr(self.xception, 'fc'):
            self.xception.fc = nn.Identity()
        else:
            self.xception.reset_classifier(0)

        self.lstm = nn.LSTM(
            input_size=2048,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1),
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2),
        )
        self.dropout = nn.Dropout(dropout)
        self.seq_len = seq_len

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = []
        for t in range(x.shape[1]):
            feat = self.xception(x[:, t])
            feat = self.dropout(feat)
            features.append(feat)
        seq_feat = torch.stack(features, dim=1)
        lstm_out, _ = self.lstm(seq_feat)
        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        pooled = torch.sum(attn_weights * lstm_out, dim=1)
        return self.classifier(pooled)

model = VideoModel(seq_len=CFG['seq_len']).to(device)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f'Total parameters: {total_params:.1f}M')
print(f'Trainable parameters: {trainable_params:.1f}M')

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
Total parameters: 27.2M
Trainable parameters: 27.2M


In [10]:
# ============================================================
# CELL 5: TRAINING HELPERS
# ============================================================
def build_optimizer(model: nn.Module):
    params = filter(lambda p: p.requires_grad, model.parameters())
    name = CFG['optimizer_name'].lower()
    if name == 'adam':
        return optim.Adam(params, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    if name == 'adagrad':
        return optim.Adagrad(params, lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    return optim.AdamW(params, lr=CFG['lr'], weight_decay=CFG['weight_decay'])

def compute_metrics(y_true: list[int], y_pred: list[int], y_prob: list[float] | None = None) -> dict[str, float]:
    acc = accuracy_score(y_true, y_pred) if y_true else 0.0
    prec_macro = precision_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0
    rec_macro = recall_score(y_true, y_pred, average='macro', zero_division=0) if y_true else 0.0
    real_precision = precision_score(y_true, y_pred, pos_label=0, zero_division=0) if y_true else 0.0
    fake_precision = precision_score(y_true, y_pred, pos_label=1, zero_division=0) if y_true else 0.0
    real_recall = recall_score(y_true, y_pred, pos_label=0, zero_division=0) if y_true else 0.0
    fake_recall = recall_score(y_true, y_pred, pos_label=1, zero_division=0) if y_true else 0.0
    metrics = {
        'accuracy': float(acc),
        'macro_precision': float(prec_macro),
        'macro_recall': float(rec_macro),
        'real_precision': float(real_precision),
        'fake_precision': float(fake_precision),
        'real_recall': float(real_recall),
        'fake_recall': float(fake_recall),
        'recall_gap_abs': float(abs(real_recall - fake_recall)),
    }
    if y_prob is not None and len(set(y_true)) > 1:
        try:
            metrics['auc'] = float(roc_auc_score(y_true, y_prob))
        except Exception:
            metrics['auc'] = 0.0
    else:
        metrics['auc'] = 0.0
    return metrics

def collect_predictions(model: nn.Module, loader: DataLoader, threshold: float = 0.5):
    model.eval()
    y_true: list[int] = []
    y_pred: list[int] = []
    y_prob: list[float] = []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = (prob >= threshold).long()
            y_true.extend(yb.detach().cpu().numpy().tolist())
            y_pred.extend(pred.detach().cpu().numpy().tolist())
            y_prob.extend(prob.detach().cpu().numpy().tolist())
    return y_true, y_pred, y_prob

def set_backbone_trainable(model: nn.Module, trainable: bool) -> None:
    for param in model.xception.parameters():
        param.requires_grad = trainable

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, CFG['pos_weight']], device=device))
optimizer = build_optimizer(model)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

# Torch AMP compatibility across versions
if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    def autocast_ctx(enabled: bool):
        return torch.amp.autocast(device_type='cuda', enabled=enabled)
else:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    def autocast_ctx(enabled: bool):
        return torch.cuda.amp.autocast(enabled=enabled)

best_state = None
best_score = float('-inf')
best_threshold = CFG['threshold']
best_metrics = None
history = []

In [11]:
# ============================================================
# CELL 6: TRAIN, EVALUATE, SAVE CHECKPOINT
# ============================================================
print('Starting training...')
print('=' * 80)

for epoch in range(1, CFG['epochs'] + 1):
    if epoch <= CFG['freeze_backbone_epochs']:
        set_backbone_trainable(model, False)
    else:
        set_backbone_trainable(model, True)

    model.train()
    running_loss = 0.0

    for step, (xb, yb) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch}/{CFG["epochs"]} [train]')):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        if epoch == 1 and step == 0:
            model_device = next(model.parameters()).device
            print(f'Runtime device check -> model: {model_device}, batch: {xb.device}')
            if torch.cuda.is_available():
                print(f'GPU memory allocated: {torch.cuda.memory_allocated() / (1024**2):.1f} MB')

        optimizer.zero_grad(set_to_none=True)
        with autocast_ctx(use_amp):
            logits = model(xb)
            loss = criterion(logits, yb)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.item())

    train_loss = running_loss / max(1, len(train_loader))

    val_true, val_pred, val_prob = collect_predictions(model, val_loader, threshold=CFG['threshold'])
    val_metrics = compute_metrics(val_true, val_pred, val_prob)
    val_loss = 1.0 - val_metrics['macro_recall']
    scheduler.step(val_loss)

    score = 0.4 * val_metrics['accuracy'] + 0.3 * val_metrics['macro_precision'] + 0.3 * val_metrics['macro_recall']
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss_proxy': val_loss,
        **val_metrics,
        'score': score,
        'lr': optimizer.param_groups[0]['lr'],
    })

    print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_acc={val_metrics['accuracy']:.4f} val_macro_p={val_metrics['macro_precision']:.4f} val_macro_r={val_metrics['macro_recall']:.4f} score={score:.4f}")
    print(f"  real_p/r={val_metrics['real_precision']:.4f}/{val_metrics['real_recall']:.4f} | fake_p/r={val_metrics['fake_precision']:.4f}/{val_metrics['fake_recall']:.4f} | gap={val_metrics['recall_gap_abs']:.4f}")
    print(f"  lr={optimizer.param_groups[0]['lr']:.6f}")

    if score > best_score:
        best_score = score
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_threshold = CFG['threshold']
        best_metrics = val_metrics.copy()
        checkpoint_path = MODELS_DIR / CFG['model_name']
        torch.save({
            'epoch': epoch,
            'model_state_dict': best_state,
            'optimizer_state_dict': optimizer.state_dict(),
            'cfg': CFG,
            'val_metrics': best_metrics,
            'score': best_score,
        }, checkpoint_path)
        print(f'  Saved best checkpoint: {checkpoint_path}')

    print('-' * 80)

if best_state is not None:
    model.load_state_dict(best_state)

test_true, test_pred, test_prob = collect_predictions(model, test_loader, threshold=best_threshold)
test_metrics = compute_metrics(test_true, test_pred, test_prob)
cm = confusion_matrix(test_true, test_pred)

print('FINAL TEST RESULTS')
print('=' * 80)
print(json.dumps(test_metrics, indent=2))
print('Confusion matrix:')
print(cm)

report = {
    'dataset_root': str(DATASET_ROOT),
    'cfg': CFG,
    'best_val_metrics': best_metrics,
    'best_score': float(best_score),
    'test_metrics': test_metrics,
    'confusion_matrix': cm.tolist(),
    'checkpoint': str(MODELS_DIR / CFG['model_name']),
    'history': history,
}

report_path = OUTPUTS_DIR / 'video_kaggle_training_report.json'
with report_path.open('w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

print(f'Report saved to: {report_path}')
print(f'Checkpoint saved to: {MODELS_DIR / CFG["model_name"]}')

Starting training...


Epoch 1/7 [train]:   0%|          | 0/2258 [00:00<?, ?it/s]

Runtime device check -> model: cuda:0, batch: cuda:0
GPU memory allocated: 129.7 MB
Epoch 1: train_loss=0.6912 val_acc=0.4871 val_macro_p=0.2435 val_macro_r=0.5000 score=0.4179
  real_p/r=0.0000/0.0000 | fake_p/r=0.4871/1.0000 | gap=1.0000
  lr=0.000050
  Saved best checkpoint: /kaggle/working/models/video_xception_bilstm_kaggle.pth
--------------------------------------------------------------------------------


Epoch 2/7 [train]:   0%|          | 0/2258 [00:00<?, ?it/s]

Epoch 2: train_loss=0.9599 val_acc=0.5336 val_macro_p=0.6786 val_macro_r=0.5448 score=0.5805
  real_p/r=0.8462/0.1109 | fake_p/r=0.5111/0.9788 | gap=0.8679
  lr=0.000050
  Saved best checkpoint: /kaggle/working/models/video_xception_bilstm_kaggle.pth
--------------------------------------------------------------------------------


...

Epoch 5: train_loss=1.1425 val_acc=0.6949 val_macro_p=0.6958 val_macro_r=0.6956 score=0.6954
  real_p/r=0.7171/0.6694 | fake_p/r=0.6746/0.7219 | gap=0.0525
  lr=0.000050
--------------------------------------------------------------------------------


Epoch 6/7 [train]:   0%|          | 0/2258 [00:00<?, ?it/s]

Epoch 6: train_loss=0.9582 val_acc=0.8046 val_macro_p=0.8045 val_macro_r=0.8047 score=0.8046
  real_p/r=0.8165/0.7984 | fake_p/r=0.7925/0.8110 | gap=0.0127
  lr=0.000050
  Saved best checkpoint: /kaggle/working/models/video_xception_bilstm_kaggle.pth
--------------------------------------------------------------------------------


Epoch 7/7 [train]:   0%|          | 0/2258 [00:00<?, ?it/s]

Epoch 7: train_loss=0.7826 val_acc=0.7404 val_macro_p=0.8022 val_macro_r=0.7462 score=0.7607
  real_p/r=0.9455/0.5242 | fake_p/r=0.6590/0.9682 | gap=0.4440
  lr=0.000050
--------------------------------------------------------------------------------
FINAL TEST RESULTS
{
  "accuracy": 0.8099173553719008,
  "macro_precision": 0.8105357600615227,
  "macro_recall": 0.8104667851284855,
  "real_precision": 0.8319148936170213,
  "fake_precision": 0.7891566265060241,
  "real_recall": 0.7883064516129032,
  "fake_recall": 0.8326271186440678,
  "recall_gap_abs": 0.04432066703116455,
  "auc": 0.8923912486331329
}
Confusion matrix:
[[391 105]
 [ 79 393]]
Report saved to: /kaggle/working/outputs/video_kaggle_training_report.json
Checkpoint saved to: /kaggle/working/models/video_xception_bilstm_kaggle.pth


In [12]:
# ============================================================
# CELL 7: CLASS-WISE METRICS (REAL VS FAKE)
# ============================================================
if 'test_true' not in globals() or 'test_pred' not in globals():
    raise RuntimeError('Run Cell 6 first to generate test_true and test_pred.')

cm2 = confusion_matrix(test_true, test_pred, labels=[0, 1])
tn, fp, fn, tp = cm2.ravel()

# Class-wise precision/recall
real_precision = precision_score(test_true, test_pred, pos_label=0, zero_division=0)
real_recall = recall_score(test_true, test_pred, pos_label=0, zero_division=0)
fake_precision = precision_score(test_true, test_pred, pos_label=1, zero_division=0)
fake_recall = recall_score(test_true, test_pred, pos_label=1, zero_division=0)

# Class-wise accuracy = correctly predicted samples of that class / total samples of that class
real_total = tn + fp
fake_total = tp + fn
real_accuracy = (tn / real_total) if real_total > 0 else 0.0
fake_accuracy = (tp / fake_total) if fake_total > 0 else 0.0

print('CLASS-WISE TEST METRICS')
print('=' * 80)
print(f"REAL  -> precision: {real_precision:.4f} | recall: {real_recall:.4f} | accuracy: {real_accuracy:.4f}")
print(f"FAKE  -> precision: {fake_precision:.4f} | recall: {fake_recall:.4f} | accuracy: {fake_accuracy:.4f}")
print('-' * 80)
print('Confusion matrix with label order [real=0, fake=1]:')
print(cm2)

CLASS-WISE TEST METRICS
REAL  -> precision: 0.8319 | recall: 0.7883 | accuracy: 0.7883
FAKE  -> precision: 0.7892 | recall: 0.8326 | accuracy: 0.8326
--------------------------------------------------------------------------------
Confusion matrix with label order [real=0, fake=1]:
[[391 105]
 [ 79 393]]
